# Q2 — Treatment effect: AAV9 vs LICA1 (micro-dystrophin gene delivery)

**Question.** Both AAV9 and LICA1 deliver the same micro-dystrophin transgene at the same dose (myotropic serotype LICA1 vs AAV9). Does LICA1 restore more than AAV9, and *where* (which fiber states, which markers, which correlations)?

**Data & methods.** Same as Q1: QUA, 111,403 matched fibers, unbiased clustering (morphology + HE only -> PCA30 -> UMAP -> GMM k=6) loaded from the precomputed cache (`scripts/build_analysis_bundle.py`); no re-clustering.

**Caveat — slide 8 (LAMP2/LGALS3/SQSTM1).** Fiber mapping *failed* for QUAG27/28/29 (3/4 AAV9 samples) and QUAG21 (1/5 mdx): the matcher found no seed pair (see fig. 10 left). AAV9 is therefore **excluded from all slide-8 analyses** (only QUAG26 carries S8 data, 66% detection). mdx S8 stats rest on 4/5 samples.


In [1]:

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import spearmanr

REPO = Path("/DATA/F2FMatcher_DDC")
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "scripts"))
from cluster_fibers import MARKERS, feature_observed_mask

BUNDLE = REPO / "results/QUA/analysis_bundle.npz"
OUT = REPO / "visualizations/Q2"
OUT.mkdir(parents=True, exist_ok=True)

z = np.load(BUNDLE, allow_pickle=True)
Xpre = z["Xpre"].astype(np.float64)          # 807 raw features, pre-imputation (NaN = not mapped)
X_umap = z["X_umap"].astype(np.float64)
groups = z["groups"]; samples = z["samples"]
clusters = z["cluster_ids"]                   # GMM k=6 labels 0..5
cols = [str(c) for c in z["cols"]]
n = len(X_umap)
idx = {c: i for i, c in enumerate(cols)}

GROUPS = ["WT", "mdx", "AAV9", "LICA1"]
GROUP_COLORS = {"WT": "#0000C0", "mdx": "#FF6000", "AAV9": "#C0C000", "LICA1": "#008000"}
CLUSTER_COLORS = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#a65628"]
CLUSTERS = [f"C{i+1}" for i in range(6)]

def mval(name):
    """Raw (pre-imputation) marker values + per-fiber observed mask (mapped & finite)."""
    spec = MARKERS[name]
    if isinstance(spec, str):
        v = Xpre[:, idx[spec]]
    else:
        v = np.mean([Xpre[:, idx[c]] for c in spec], axis=0)
    o = feature_observed_mask(name, Xpre, cols) & np.isfinite(v)
    return v, o

def umap_scatter(ax, m, color=None, s=1.0, alpha=0.3):
    ax.scatter(X_umap[m, 0], X_umap[m, 1], s=s, alpha=alpha, c=color,
               edgecolors="none", rasterized=True)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

def rho_pair(a, b, m):
    va, oa = mval(a); vb, ob = mval(b)
    o = m & oa & ob
    if o.sum() < 30:
        return np.nan
    r, _ = spearmanr(va[o], vb[o])
    return r

print(f"fibers: {n:,}")
print(pd.Series(groups).value_counts().reindex(GROUPS))


fibers: 111,403
WT       34861
mdx      23057
AAV9     25662
LICA1    27823
Name: count, dtype: int64


## 1. Global landscape

On the unbiased UMAP, both treatments pull fibers back from the mdx-specific regions toward the WT core; LICA1's cloud overlaps WT more closely than AAV9's.


In [2]:

# Fig 1 — UMAP colored by group
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for g in GROUPS:
    umap_scatter(ax, groups == g, color=GROUP_COLORS[g])
handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=GROUP_COLORS[g],
                  markersize=8, label=f"{g} (n={int((groups == g).sum()):,})") for g in GROUPS]
ax.legend(handles=handles, fontsize=10, loc="best")
ax.set_title("QUA fibers — UMAP (morphology + HE only), colored by group", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT / "fig1_umap_by_group.png", dpi=200)
plt.show()


In [3]:

# Fig 2 — per-group UMAP panels colored by cluster (spatial view of the state shift)
fig, axes = plt.subplots(2, 2, figsize=(9, 8), sharex=True, sharey=True)
for ax, g in zip(axes.ravel(), GROUPS):
    m = groups == g
    for c in range(6):
        umap_scatter(ax, m & (clusters == c), color=CLUSTER_COLORS[c])
    ax.text(0.05, 0.95, f"{g} (n={int(m.sum()):,})", transform=ax.transAxes,
            fontsize=12, fontweight="bold", ha="left", va="top")
fig.tight_layout()
fig.savefig(OUT / "fig2_umap_groups_clusters.png", dpi=200)
plt.show()


## 2. Cluster composition shift

Treatment response is *cluster-specific* (in-group fraction, i.e. % of each group's fibers in the cluster):

- **C2** (small fibrotic/inflammatory): mdx 15.0% -> AAV9 8.4% -> **LICA1 6.6%** (cleared, LICA1 more)
- **C3** (small active + immune): mdx 37.6% -> AAV9 27.0% -> **LICA1 20.8%** (cleared, LICA1 more)
- **C4** (small healthy fast): mdx 4.0% -> AAV9 12.1% -> **LICA1 14.1%** (recovered, LICA1 more)
- **C6** (large quiet WT-like): mdx 7.3% -> AAV9 18.6% -> **LICA1 23.9%** (recovered, LICA1 more)
- **C5** (large Myh4/BM-defect): mdx 20.7% -> AAV9 21.8% -> LICA1 21.6% (**unchanged = resistant**)
- **C1** (large quiet mixed): 15.4% -> 12.1% / 13.0% (mildly reduced)


In [4]:

# Fig 3 — cluster composition shift mdx -> AAV9 -> LICA1 (in-group fraction, stacked)
rows = []
for c in range(6):
    mc = clusters == c
    for g in GROUPS:
        rows.append(dict(cluster=CLUSTERS[c], group=g,
                         pct=100 * (mc & (groups == g)).sum() / (groups == g).sum()))
comp = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(6, 5))
x = np.arange(3)
bottom = np.zeros(3)
for c in range(6):
    d = comp[comp.cluster == CLUSTERS[c]].set_index("group").loc[["mdx", "AAV9", "LICA1"], "pct"].values
    ax.bar(x, d, bottom=bottom, width=0.55, label=CLUSTERS[c],
           color=CLUSTER_COLORS[c], edgecolor="white", linewidth=0.5)
    for i, (v, b) in enumerate(zip(d, bottom)):
        if v > 4:
            ax.text(i, b + v / 2, f"{v:.0f}", ha="center", va="center", fontsize=8, color="white")
    bottom += d
ax.set_xticks(x, ["mdx", "AAV9", "LICA1"], fontsize=12)
ax.set_ylabel("% of group's fibers")
ax.set_title("Cluster composition: mdx -> AAV9 -> LICA1", fontweight="bold")
ax.legend(fontsize=9, ncols=2)
fig.tight_layout()
fig.savefig(OUT / "fig3_composition_shift.png", dpi=200)
plt.show()
comp.pivot(index="cluster", columns="group", values="pct").round(1)[["mdx", "AAV9", "LICA1", "WT"]]


group,mdx,AAV9,LICA1,WT
cluster,,,,
C1,15.4,12.1,13.0,10.2
C2,15.0,8.4,6.6,3.6
C3,37.6,27.0,20.8,3.8
C4,4.0,12.1,14.1,36.7
C5,20.7,21.8,21.6,0.5
C6,7.3,18.6,23.9,45.1


## 3. Dystrophin restoration — the functional readout

Both treatments restore membrane dystrophin, but **LICA1 clearly outperforms AAV9 in every cluster** (medians 53-70 vs 36-48; WT 47-55). LICA1 even *exceeds* WT levels in C3/C4/C5/C6. In dystrophin-deficient-fraction terms (below global WT p25): mdx 79-99% -> AAV9 41-68% -> **LICA1 10-36%**, i.e. LICA1 reaches (or beats) the WT baseline (14-35%) while AAV9 stays intermediate. The effect is consistent across all 5 LICA1 animals (fig. 5).


In [5]:

# Fig 4 — Dystrophin (membrane) restoration per cluster: medians (left), % deficient (right)
dv, do = mval("Dystrophin")
thr = np.percentile(dv[do & (groups == "WT")], 25)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ax = axes[0]
for k, g in enumerate(GROUPS):
    med = [np.median(dv[(clusters == c) & (groups == g) & do]) for c in range(6)]
    ax.bar(np.arange(6) + (k - 1.5) * 0.2, med, width=0.2, label=g,
           color=GROUP_COLORS[g], edgecolor="black", linewidth=0.4)
ax.set_xticks(range(6), CLUSTERS, fontsize=11)
ax.set_ylabel("median dystrophin (0-255, membrane)")
ax.set_title("Dystrophin per cluster", fontweight="bold")
ax.legend(fontsize=9)

ax = axes[1]
for k, g in enumerate(GROUPS):
    frac = [100 * np.mean(dv[(clusters == c) & (groups == g) & do] < thr) for c in range(6)]
    ax.bar(np.arange(6) + (k - 1.5) * 0.2, frac, width=0.2, label=g,
           color=GROUP_COLORS[g], edgecolor="black", linewidth=0.4)
ax.set_xticks(range(6), CLUSTERS, fontsize=11)
ax.set_ylabel("% of fibers below WT p25")
ax.set_title(f"Dystrophin-deficient fibers (threshold = WT p25 = {thr:.0f})", fontweight="bold")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(OUT / "fig4_dystrophin_restoration.png", dpi=200)
plt.show()


In [6]:

# Fig 5 — per-sample median dystrophin (consistency across animals)
dv, do = mval("Dystrophin")
ps = pd.DataFrame({"sample": samples, "group": groups,
                   "dys": np.where(do, dv, np.nan)})
ps = ps.groupby(["sample", "group"])["dys"].median().reset_index()
order = {g: i for i, g in enumerate(GROUPS)}
ps["o"] = ps.group.map(order)
ps = ps.sort_values(["o", "sample"])
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(range(len(ps)), ps.dys, color=[GROUP_COLORS[g] for g in ps.group],
       edgecolor="black", linewidth=0.3)
ax.set_xticks(range(len(ps)), [f"{s}\n({g[0]})" for s, g in zip(ps["sample"], ps["group"])], fontsize=8)
ax.set_ylabel("median membrane dystrophin (0-255)")
ax.set_title("Per-sample dystrophin (5 WT, 5 mdx, 4 AAV9, 5 LICA1)", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT / "fig5_dystrophin_per_sample.png", dpi=200)
plt.show()
ps.round(1)


,sample,group,dys,o
0,QUAG01,WT,47.1,0
1,QUAG02,WT,56.0,0
2,QUAG03,WT,53.8,0
3,QUAG04,WT,51.5,0
4,QUAG05,WT,40.4,0
5,QUAG21,mdx,23.3,1
6,QUAG22,mdx,30.4,1
7,QUAG23,mdx,36.5,1
8,QUAG24,mdx,32.5,1
9,QUAG25,mdx,30.2,1


## 4. Marker-level treatment effect (LICA1 vs AAV9)

LICA1 restores more than AAV9: **SQSTM1, dystrophin, Myh2, Myh4, COX, HE, Laminin** (Cliff up to 0.48). AAV9 leaves more residual **IgG, LAMP2, CD11b, NADH, Myh7** — i.e. more residual inflammation/lysosomal stress and less fiber-type normalization. The 4-panel z map shows treated clusters moving toward the WT pattern, with LICA1 panels closest to WT.


In [7]:

# Fig 6 — AAV9 vs LICA1 per cluster x marker: Cliff's delta (red = higher in LICA1)
MARK = list(MARKERS)
mat = np.full((6, len(MARK)), np.nan)
for i in range(6):
    mc = clusters == i
    for j, name in enumerate(MARK):
        v, o = mval(name)
        x = v[o & mc & (groups == "AAV9")]
        y = v[o & mc & (groups == "LICA1")]
        if len(x) < 10 or len(y) < 10:
            continue
        less = (x[:, None] < y[None, :]).sum()
        great = (x[:, None] > y[None, :]).sum()
        mat[i, j] = (great - less) / (len(x) * len(y))

fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(mat, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(MARK)), MARK, rotation=60, fontsize=10)
ax.set_yticks(range(6), CLUSTERS, fontsize=11)
for i in range(6):
    for j in range(len(MARK)):
        if np.isfinite(mat[i, j]):
            ax.text(j, i, f"{mat[i, j]:+.2f}", ha="center", va="center", fontsize=7,
                    color="black" if abs(mat[i, j]) < 0.6 else "white")
ax.set_title("LICA1 vs AAV9 — Cliff's delta per cluster (red = higher in LICA1)", fontweight="bold")
fig.colorbar(im, label="Cliff's delta")
fig.tight_layout()
fig.savefig(OUT / "fig6_cliff_lica1_aav9.png", dpi=200)
plt.show()


In [8]:

# Fig 7 — marker z per cluster, 4 group panels (global z; WT = reference)
MARK = list(MARKERS)
zmat = np.full((4, 6, len(MARK)), np.nan)
for j, name in enumerate(MARK):
    v, o = mval(name)
    mu, sd = np.nanmean(v[o]), np.nanstd(v[o])
    zv = np.full(n, np.nan)
    zv[o] = (v[o] - mu) / sd
    for gi, g in enumerate(GROUPS):
        for c in range(6):
            zmat[gi, c, j] = np.nanmean(zv[(clusters == c) & (groups == g)])

fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
for ax, gi, g in zip(axes.ravel(), range(4), GROUPS):
    im = ax.imshow(zmat[gi], aspect="auto", cmap="RdBu_r", vmin=-1.5, vmax=1.5)
    ax.set_xticks(range(len(MARK)), MARK, rotation=60, fontsize=8)
    ax.set_yticks(range(6), CLUSTERS, fontsize=10)
    ax.set_title(g, fontsize=13, fontweight="bold", color=GROUP_COLORS[g])
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle("Marker z (global) per cluster, per group", fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(OUT / "fig7_marker_z_groups.png", dpi=200)
plt.show()


## 5. Correlations restored by treatment

Correlations lost in mdx (Q1) are partially restored, and LICA1 generally restores more:

- **Dystrophin~area** (mdx -0.22): restored by AAV9 (-0.02, at WT level) and partially by LICA1 (-0.14)
- **Myh7~NADH** (mdx 0.67): both restore (0.78 / 0.76 vs WT 0.88)
- **NADH~area** (mdx -0.56): both restore (-0.69 / -0.72 vs WT -0.80)
- **LAMP2~SQSTM1** (mdx 0.70): **only LICA1 restores** it (0.85, above WT 0.80); AAV9 *worsens* it (0.59)
- **Laminin~WGA** (BM-membrane coupling) is **not restored by either** (0.11 -> 0.06 / -0.00)

Per cluster (fig. 9): LICA1 restores Myh7~COX in C4 (0.22 -> 0.41) and Myh7~NADH + NADH~COX even in the resistant C5 (0.27 -> 0.47 and 0.66 -> 0.79) — LICA1 re-couples metabolism in C5 even though it does not change C5's composition.


In [9]:

# Fig 8 — correlation restoration: Spearman rho per group for key pairs
PAIRS = [("Myh7", "NADH"), ("NADH", "COX"), ("Myh7", "COX"), ("NADH", "area"),
         ("DAPI", "area"), ("Laminin", "WGA"), ("Dystrophin", "area"), ("LAMP2", "SQSTM1")]
names = [f"{a}~{b}" for a, b in PAIRS]
R = np.array([[rho_pair(a, b, groups == g) for g in GROUPS] for a, b in PAIRS])

fig, ax = plt.subplots(figsize=(8, 5))
w = 0.2
y = np.arange(len(PAIRS))
for k, g in enumerate(GROUPS):
    ax.bar(y + (k - 1.5) * w, R[:, k], width=w, label=g, color=GROUP_COLORS[g],
           edgecolor="black", linewidth=0.3)
ax.axvline(0, color="0.7", lw=0.8)
ax.set_yticks(y, names, fontsize=10)
ax.set_xlabel("Spearman rho (all fibers)")
ax.set_title("Correlations per group (blue WT = healthy reference)", fontweight="bold")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(OUT / "fig8_corr_restoration.png", dpi=200)
plt.show()
pd.DataFrame(R, index=names, columns=GROUPS).round(2)


,WT,mdx,AAV9,LICA1
Myh7~NADH,0.88,0.67,0.78,0.76
NADH~COX,0.90,0.86,0.85,0.89
Myh7~COX,0.82,0.71,0.69,0.74
NADH~area,-0.80,-0.56,-0.69,-0.72
DAPI~area,-0.61,-0.39,-0.46,-0.37
Laminin~WGA,0.27,0.11,0.06,-0.00
Dystrophin~area,-0.07,-0.22,-0.02,-0.14
LAMP2~SQSTM1,0.80,0.69,0.58,0.85


In [10]:

# Fig 9 — per-cluster correlation structure (rho, cluster x group) for 4 key pairs
sel = [("Myh7", "NADH"), ("NADH", "COX"), ("Dystrophin", "area"), ("LAMP2", "SQSTM1")]
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, (a, b) in zip(axes.ravel(), sel):
    mat = np.array([[rho_pair(a, b, (clusters == c) & (groups == g)) for g in GROUPS]
                    for c in range(6)])
    im = ax.imshow(mat, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(4), GROUPS, fontsize=10)
    ax.set_yticks(range(6), CLUSTERS, fontsize=10)
    for i in range(6):
        for j in range(4):
            if np.isfinite(mat[i, j]):
                ax.text(j, i, f"{mat[i, j]:+.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(f"{a} ~ {b}", fontweight="bold")
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle("Per-cluster Spearman rho by group", fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(OUT / "fig9_corr_per_cluster.png", dpi=200)
plt.show()


In [11]:

# Fig 10 — slide 8 (lysosomal/autophagy): AAV9 EXCLUDED (mapping failed for 3/4 AAV9 samples)
# left: per-sample S8 detection rate; right: per-cluster boxplots WT/mdx/LICA1
from plot_pca_all_features import slide_channel_blocks
st8, nc8 = slide_channel_blocks()[8]
s8 = Xpre[:, 15 + st8 * 36:15 + (st8 + nc8) * 36]
s8obs = (~np.isnan(s8)).any(axis=1)
det = pd.DataFrame({"sample": samples, "group": groups, "s8": s8obs})
det = det.groupby(["sample", "group"])["s8"].mean().unstack("group").round(2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [1, 1.6]})
ax = axes[0]
im = ax.imshow(det.values, cmap="YlOrRd", aspect="auto")
nsamp = {g: len(set(samples[groups == g])) for g in GROUPS}
ax.set_xticks(range(4), [f"{g}\n({nsamp[g]} samples)" for g in GROUPS], fontsize=9)
ax.set_yticks(range(len(det)), det.index, fontsize=8)
for i in range(det.shape[0]):
    for j in range(4):
        v = det.values[i, j]
        if np.isfinite(v):
            ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=8)
ax.set_title("S8 mapping rate per sample", fontweight="bold")
fig.colorbar(im, ax=ax, label="fraction of fibers mapped on S8")

ax = axes[1]
S8M = ["LAMP2", "LGALS3", "SQSTM1"]
for k, name in enumerate(S8M):
    v, o = mval(name)
    data = [v[(clusters == c) & (groups == g) & o] for c in range(6) for g in ["WT", "mdx", "LICA1"]]
    bp = ax.boxplot(data, positions=np.arange(18) + np.random.default_rng(0).uniform(-0.05, 0.05, 18),
                    widths=0.6, showfliers=False, patch_artist=True,
                    boxprops=dict(linewidth=0.5), medianprops=dict(color="black"))
    for i, (c, g) in enumerate([(c, g) for c in range(6) for g in ["WT", "mdx", "LICA1"]]):
        bp["boxes"][i].set_facecolor(GROUP_COLORS[g])
        bp["boxes"][i].set_alpha(0.7)
    ymax = max(np.nanmax(d) for d in data)
    ax.text(k * 6 + 2.5, ymax * 1.06, name, ha="center", fontsize=11, fontweight="bold")
ax.set_xticks(np.arange(6) * 6 + 2.5, CLUSTERS, fontsize=10)
ax.set_ylabel("intensity (0-255)")
ax.set_title("Lysosomal/autophagy markers per cluster (AAV9 excluded)", fontweight="bold")
handles = [Line2D([0], [0], marker="s", color="w", markerfacecolor=GROUP_COLORS[g], markersize=10, label=g)
           for g in ["WT", "mdx", "LICA1"]]
ax.legend(handles=handles, fontsize=9, loc="upper right")
fig.tight_layout()
fig.savefig(OUT / "fig10_slide8_lysosomal.png", dpi=200)
plt.show()
det


group,AAV9,LICA1,WT,mdx
sample,,,,
QUAG01,NaN,NaN,0.26,NaN
QUAG02,NaN,NaN,0.63,NaN
QUAG03,NaN,NaN,0.84,NaN
QUAG04,NaN,NaN,0.63,NaN
QUAG05,NaN,NaN,0.55,NaN
QUAG21,NaN,NaN,NaN,0.00
QUAG22,NaN,NaN,NaN,0.69
QUAG23,NaN,NaN,NaN,0.74
QUAG24,NaN,NaN,NaN,0.75


## 6. C5 deep dive — the resistant cluster

C5 (large, Myh4-enriched, basement-membrane defect, metabolically quiet) is the only cluster whose composition does not change with treatment (20.7% -> 21.8% / 21.6%). Yet **within C5, LICA1 still restores dystrophin** (median 29 -> 53, above WT 49) and re-couples Myh7~NADH — the transgene works in C5 fibers; what resists is the *morphological/HE phenotype* that defines the cluster (size, Myh4, HE pattern). Likely contributors [inferred]: BM defect (Laminin down) impairs uptake, large size lowers surface-to-volume transduction efficiency, and the quiet metabolic state limits response capacity.


In [12]:

# Fig 11 — C5 deep dive (the treatment-resistant cluster)
c5 = clusters == 4
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[0]
for g in GROUPS:
    umap_scatter(ax, c5 & (groups == g), color=GROUP_COLORS[g])
handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=GROUP_COLORS[g], markersize=8,
                  label=f"{g} ({int((c5 & (groups == g)).sum()):,})") for g in GROUPS]
ax.legend(handles=handles, fontsize=9, loc="best")
ax.set_title("C5 fibers (all groups)", fontweight="bold")

ax = axes[1]
mk = ["Dystrophin", "Laminin", "Myh4", "NADH", "COX", "IgG"]
w = 0.25
for k, g in enumerate(["mdx", "AAV9", "LICA1"]):
    med = [np.median(mval(m)[0][mval(m)[1] & c5 & (groups == g)]) for m in mk]
    ax.bar(np.arange(len(mk)) + (k - 1) * w, med, width=w, label=g,
           color=GROUP_COLORS[g], edgecolor="black", linewidth=0.4)
ax.set_xticks(range(len(mk)), mk, fontsize=10)
ax.set_ylabel("median (0-255)")
ax.set_title("C5 marker medians", fontweight="bold")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(OUT / "fig11_C5_deep_dive.png", dpi=200)
plt.show()


## Answer to Q2 — is LICA1 better than AAV9?

**Yes, consistently and across every fiber state:**

1. **Dystrophin**: LICA1 median 60 vs AAV9 39 (WT 50) — LICA1 exceeds WT in 4/6 clusters; dystrophin-deficient fraction 10-36% (LICA1) vs 41-68% (AAV9) vs 79-99% (mdx).
2. **Fiber-state composition**: both clear the pathological C2/C3 and recover C4/C6, but LICA1 does it more (C2 6.6 vs 8.4%; C3 20.8 vs 27.0%; C4 14.1 vs 12.1%; C6 23.9 vs 18.6%).
3. **Markers**: LICA1 normalizes more (SQSTM1, Myh2/Myh4 fiber type, COX, Laminin) and leaves less residual inflammation/lysosomal stress (IgG, CD11b, LAMP2, NADH lower in LICA1).
4. **Correlations**: LICA1 restores the lysosome-autophagy coupling (LAMP2~SQSTM1) that AAV9 worsens, and re-couples metabolism (Myh7~NADH, NADH~COX) in more clusters, including resistant C5.
5. **Limits of both treatments**: the resistant C5 state (21-22% of treated fibers) and the BM-membrane coupling (Laminin~WGA) are not corrected by either serotype; slide-8 data confirm LICA1 normalizes lysosomal stress (LAMP2 47 -> 41 vs WT 36) in the groups where mapping worked.
